In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "mCRSP.csv")
OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "monthlyCRSP.parquet")

In [ ]:
#1 Load CRSP Data

SQL = """
SELECT
    a.permno,
    a.permco,
    a.date,
    a.ret,
    a.retx,
    a.vol,
    a.shrout,
    a.prc,
    a.cfacshr,
    a.bidlo,
    a.askhi,
    b.shrcd,
    b.exchcd,
    b.siccd,
    b.ticker,
    b.shrcls,
    c.dlstcd,
    c.dlret
FROM crsp.msf AS a
LEFT JOIN crsp.msenames AS b
  ON a.permno = b.permno
 AND b.namedt <= a.date
 AND a.date   <= b.nameendt
LEFT JOIN crsp.msedelist AS c
  ON a.permno = c.permno
 AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE a.date >= DATE '2000-01-01';
"""

In [ ]:
#2 CRSP Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["date"])  # 'dlstdt' already handled in SQL

In [2]:
#3 Data Cleaning

# ---------------- Make 2-digit SIC ----------------
# Stata: rename siccd sicCRSP; tostring; gen sic2D = substr(...,1,2); destring
df = df.rename(columns={"siccd": "sicCRSP"})
# to string, left-pad if needed, then first 2 chars; coerce to numeric
sic_str = df["sicCRSP"].astype("Int64").astype(str).str.replace("<NA>", "", regex=False)
sic_str = sic_str.where(sic_str != "", other=np.nan)
df["sic2D"] = pd.to_numeric(sic_str.str[:2], errors="coerce").astype("Int64")

# ---------------- Monthly time index ----------------
# Stata: gen time_avail_m = mofd(date); format %tm; drop date
df["time_avail_m"] = df["date"].dt.to_period("M").dt.to_timestamp("MS")
df = df.drop(columns=["date"])

# ---------------- Delisting return adjustments ----------------
# Rules:
# dlret = -0.35 if missing & (dlstcd = 500 or 520–584) & (exchcd in 1,2)
# dlret = -0.55 if missing & (dlstcd = 500 or 520–584) & exchcd = 3
# dlret = -1 if dlret < -1 (cap)
# dlret = 0 if still missing
dlret = df["dlret"].copy()

cond_dlst = (df["dlstcd"].eq(500)) | (df["dlstcd"].between(520, 584, inclusive="both"))

mask1 = dlret.isna() & cond_dlst & df["exchcd"].isin([1, 2])
dlret = dlret.where(~mask1, -0.35)

mask2 = dlret.isna() & cond_dlst & df["exchcd"].eq(3)
dlret = dlret.where(~mask2, -0.55)

# Cap at -1
dlret = dlret.where(~(dlret < -1), -1)

# Set remaining missing to 0
dlret = dlret.fillna(0.0)

df["dlret"] = dlret

# ---------------- Combine returns with delisting returns ----------------
# Updated rule (2022-02): ret = (1+ret)*(1+dlret) - 1
# If ret is missing but dlret != 0, set ret = dlret
ret = df["ret"].copy()

# Combine when ret is not missing
mask_ret_notna = ret.notna()
ret.loc[mask_ret_notna] = (1.0 + ret.loc[mask_ret_notna]) * (1.0 + dlret.loc[mask_ret_notna]) - 1.0

# If ret is missing and dlret != 0, set ret = dlret
mask_take_dlret = ret.isna() & (dlret != 0)
ret = ret.where(~mask_take_dlret, dlret)

df["ret"] = ret

# ---------------- Unit conversions ----------------
# Stata: shrout = shrout/1000 ; vol = vol/10^4
df["shrout"] = df["shrout"] / 1000.0
df["vol"]    = df["vol"] / (10.0**4)

# ---------------- Market cap (mve_c) ----------------
# mve_c = shrout * abs(prc)
df["mve_c"] = df["shrout"] * df["prc"].abs()

# ---------------- Housekeeping ----------------
# Drop dlret dlstcd permco (as in Stata)
df = df.drop(columns=["dlret", "dlstcd", "permco"], errors="ignore")

# Optional: compress dtypes a bit
for col in ["permno", "shrcd", "exchcd", "sicCRSP", "sic2D", "cfacshr"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")

# ---------------- Save ----------------
# CSV (for your IO-Momentum in R) and Parquet (faster reloads)
df.to_csv(OUT_CSV, index=False)
df.to_parquet(OUT_PARQUET, index=False)

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_PARQUET)
print(df.head())